# Neural processing v5 — SOMA-SELECTIVE ROI acceptance (Aug 9 PM)
**Pipeline order is unchanged from v4: raw → 1-D temporal Gaussian filter → motion
correction → PCA denoising → binned detection → ROIs → traces → ΔF/F.**

## What changed from v4, and why
v4 returned mostly **dendrites, not somata**. Three separate causes, all in the ROI
*acceptance* step — none in the detection or trace math, which are untouched:

1. **The size band was in pixels, with no physical meaning.** `SIZE_MIN, SIZE_MAX =
   15, 100 px` corresponds to equivalent diameters of 4.4–11.3 px. Whether that is a
   soma or a fragment depends entirely on µm/pixel, which v4 never read. At a typical
   Bruker zoom (~0.6–1.1 µm/px) that band tops out at **7–12 µm — at or below the
   *smallest* cortical soma** — so real somata were rejected for being *too large*
   while dendrite fragments landed squarely inside the band. v5 reads
   `micronsPerPixel` from the t-series XML and derives the band from a soma diameter
   in **µm**. The cell prints the v4 band in µm next to it: if the two do not overlap,
   that alone explains a dendrite-only ROI set.
2. **Area cannot tell a soma from a dendrite.** A 10 µm soma and a 35 × 2 µm dendritic
   segment have the same pixel count. v5 adds two **shape** criteria — elongation
   (`axis_ratio`) and raggedness (`solidity`) — computed per ROI from the mask itself.
3. **Greedy correlation growth is, structurally, a dendrite tracer.** Every pixel along
   a process carries the same calcium transient, so growth runs the length of the
   dendrite; a soma is bounded by its membrane and stops on its own. That behaviour is
   not a bug and v5 does not change it — it is why acceptance has to be shape-aware.

Everything else — the filter, registration, PCA, correlation map, `next_roi` growth
rule, trace extraction, ΔF/F, bleach check — is **byte-identical to v4**.

## Why this ordering (unchanged from v4)
Rationale: frame-by-frame SNR is poor (sharp noise edges), so motion-correcting raw
frames is unreliable, and PCA before MC did not work well. The 1-D temporal filter
(instructor-notebook code, VERBATIM) makes each frame a local time-average — good
registration input — then MC, then PCA on the aligned movie.

**Code-fidelity contract:** computations are the existing pipeline's, reordered — no
new math. Every deviation from verbatim is flagged in the cell that contains it.
All v3 audit fixes retained (view-mutation fix, label==row invariant, data-driven ROI
count, shift-aware border, NaN hygiene, bleach check, crash fixes).

**Three things this notebook REQUIRES of you — full explanations at the relevant cells:**
1. **Confirm the FOV scale** (µm/px) and the soma diameter band. The scale is read from
   the t-series XML; the band is biology you are asserting. See the "FOV scale" cell.
2. **Recalibrate `CORR_THRESHOLD`** for your dataset (the 1-D filter inflates the whole
   correlation map; the inherited 0.3 over-detects). See the ⚠️ cell before ROI extraction.
3. **Run it twice: `PCA_ENABLE = True` and `PCA_ENABLE = False`.** PCA inflates pairwise
   ROI correlations by construction; reported statistics need the PCA-off control.
   See the ⚠️ cell before Save.

Motion correction uses `normalization=None` in `phase_cross_correlation` — required
for this ordering (measured 1.23 px → 0.086 px RMS registration error vs skimage's
default on filtered frames).

Known properties of this ordering, stated once (team-accepted trade-offs):
- Downstream traces inherit the 1-D filter's bandwidth (σ=5 frames ≈ 0.17 s at 30 Hz →
  ~1 Hz low-pass; GCaMP7s decays are preserved, fast rises are smoothed) and, when PCA
  is enabled, the shared rank-`PCA_RANK` basis (pairwise-correlation caveat discussed
  in the PCA cell).
- The instructor kernel has 20 taps (`arange(-10,10)`) — one sample asymmetric →
  a ~half-frame (~16 ms) temporal offset. Kept verbatim; negligible at the 10 Hz
  analysis timebase, noted for sync bookkeeping.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
try:
    from tqdm import tqdm
except ImportError:                       # progress bars are cosmetic
    def tqdm(x, **k):
        return x
from scipy.stats import zscore
from scipy import ndimage
from scipy.ndimage import shift as ndshift, convolve
from skimage.registration import phase_cross_correlation
from skimage.morphology import binary_dilation
import tifffile as tiff
import xml.etree.ElementTree as ET      # v5: FOV scale from the t-series XML
import json, time
try:                                    # v5: convex hull for ROI solidity
    from scipy.spatial import ConvexHull as _HULL
except ImportError:
    _HULL = None
    print("WARNING: scipy.spatial unavailable — solidity disabled, shape filter will "
          "use elongation only.")

## Parameters — the ONLY cell you edit per run
`FRAME_PERIOD` is still set by hand from the t-series XML (directive 1 — full XML
extraction stays deferred). **New in v5:** ROI size is declared in **microns**, not
pixels; the conversion uses `micronsPerPixel` read from that same XML by the "FOV
scale" cell below. Everything else has a sane default; the binning sweep helps pick
`DETECT_BIN`.

In [ ]:
# ---- data ----
DATA_PATH = Path('/grid/courses/data/imagcourse/GECI Project Jonathons/Old Data/jonathans_finale/20260808_M1_Mouse1/spon_sniff_run1/TSeries-08082026-0955-020')
TIF_NAME  = 'TSeries-08082026-0955-020_Cycle00001_Ch2_000001.ome.tif'
FRAME_PERIOD = 0.0328181     # s/frame — from the XML, BY HAND for now (directive 1)
XML_PATH  = None             # None = auto-find the t-series XML inside DATA_PATH

# ---- pipeline steps (order is FIXED: filter -> MC -> PCA) ----
GAUSS_WIDTH = 5              # 1-D temporal Gaussian sigma, FRAMES (instructor value: 5)
PCA_ENABLE = True            # PCA denoising of the aligned movie (pipeline step 3)
PCA_RANK  = 50               # rank of the reconstruction

# ---- detection ----
# NOTE (measured, see the "threshold recalibration" cell): with the v4 ordering the
# 1-D filter ALREADY does the temporal smoothing that binning was introduced for, so
# DETECT_BIN=1 is the right default here, and CORR_THRESHOLD must be RAISED because
# the filter inflates background correlations too. 0.3 (the unfiltered value) badly
# over-detects on filtered data.
DETECT_BIN = 1               # temporal bin for DETECTION ONLY; 1 = none; "auto" = ~6 Hz
CORR_THRESHOLD = 0.5         # RECALIBRATE per dataset with the threshold cell below

# ---- ROI acceptance: ANATOMICAL, soma-selective (v5) ----------------------
# Sizes are declared in MICRONS and converted to pixels using micronsPerPixel from
# the t-series XML (see the "FOV scale" cell). This is the v5 fix for dendrite
# contamination: a pixel band means nothing until you know the pixel size.
UM_PER_PX     = None         # None = read from XML; set a float to override it
SOMA_DIAM_UM  = (9.0, 20.0)  # accepted SOMA diameter, µm. Mouse cortical pyramidal
                             # somata ~12-20 µm; interneurons ~9-15 µm. Widen the low
                             # end only if you can point at the cells you are gaining.
SIZE_GROW_MULT = 1.8         # growth cap = this x the max accepted area, so oversize
                             # objects grow far enough to be SEEN as oversize and
                             # rejected, instead of being truncated INTO the band

# Shape gate — this is what actually separates somata from processes.
SHAPE_FILTER   = True
MAX_AXIS_RATIO = 2.5         # major/minor of the equivalent ellipse. Round soma
                             # ~1.0-1.6; straight dendritic segment >3. Catches
                             # STRAIGHT processes.
MIN_SOLIDITY   = 0.75        # area / convex-hull area. Round soma ~0.85-0.95;
                             # curved or branched process <0.6. Catches BENT and
                             # BRANCHED processes (a perfectly straight line has
                             # solidity ~1 — which is why both metrics are needed).

# Runaway guard on the GROWTH LOOP (counts candidates grown, not ROIs accepted — a
# low threshold produces thousands of 1-px candidates). Each iteration consumes at
# least one seed pixel, so the loop always terminates; this only bounds the runtime.
# It WARNS if it fires, because firing means the correlation map was not exhausted
# and the ROI set is silently truncated.
MAX_CANDIDATES = 20000

# ---- QC ----
BORDER_PX_MIN = 5            # border zeroing floor; raised automatically to max shift+1
DFF_PERCENTILE = 15          # static F0 = bottom 15% of each ROI trace (team decision)
BLEACH_WARN_PCT = 20         # warn if |mean-F drift| over the run exceeds this

## Load
`frames` must be (T, Y, X). Confirm T against the XML's frame count by eye — a partial
upload silently truncates and poisons everything downstream.

In [ ]:
t0 = time.time()
frames = tiff.imread(DATA_PATH / TIF_NAME)
frames = np.squeeze(frames)           # single-channel OME sometimes loads as (T,1,Y,X)
if frames.ndim == 2:
    frames = frames[None]
assert frames.ndim == 3, (f"expected (T, Y, X), got {frames.shape} — multi-channel or "
                          f"multi-plane data needs an explicit axis choice HERE before "
                          f"anything downstream runs")
T, Y, X = frames.shape
frame_rate = 1.0 / FRAME_PERIOD
time_vector = np.arange(T) * FRAME_PERIOD   # exact spacing (v2's linspace drifted by one
                                            # frame period across the run)
print(f"movie {frames.shape} {frames.dtype}  ({T/frame_rate:.0f}s @ {frame_rate:.2f} Hz, "
      f"{frames.nbytes/1e9:.1f} GB raw, load {time.time()-t0:.0f}s)")
print("CHECK: does T match the XML <Frame> count?")

In [ ]:
original_anatomy = frames.mean(0)
plt.figure(figsize=(4, 4))
plt.imshow(original_anatomy, cmap="gray",
           vmin=np.percentile(original_anatomy, 1), vmax=np.percentile(original_anatomy, 99))
plt.title("original anatomy (raw mean)"); plt.axis("off"); plt.show()

## FOV scale (v5) — µm/pixel from the XML, and the soma size band it implies
**This is the cell that fixes the dendrite problem.** An ROI size band in *pixels* is
meaningless on its own: the same 100 px is a whole soma at 2 µm/px and a fragment of
one at 0.4 µm/px. PrairieView records the true scale in the t-series XML as
`micronsPerPixel`, so we read it and convert a **soma diameter in µm** into a pixel
area band.

`micronsPerPixel` carries an X, a Y **and a Z** entry — only X and Y are in-plane, and
using the Z value by mistake silently corrupts the scale, so they are selected by name.

The printout compares the band v4 used (15–100 px) against the band this FOV implies.
**If they do not overlap, that mismatch alone explains the dendrite-only ROI set:** v4
was rejecting every real soma for being oversized while accepting process fragments
that happened to fall in the pixel window.

Two escape hatches, both in the parameters cell: set `UM_PER_PX` to a number to bypass
the XML entirely, or point `XML_PATH` at the file if it is not beside the tif. The
cell **stops** rather than guessing a scale — a wrong µm/px would put fabricated
micron numbers on a slide.

In [ ]:
# XML reader: the micronsPerPixel branch of parse_pv_xml() from
# ca_pipeline/ca_extract.py, narrowed to what the size band needs.
def pv_fov_scale(data_path, xml_path=None):
    """-> (um_per_px | None, info dict). Averages the X and Y entries ONLY."""
    if xml_path is None:
        cands = [p for p in sorted(Path(data_path).glob("*.xml"))
                 if "Voltage" not in p.name]
        if not cands:
            return None, {"error": f"no t-series XML in {data_path}"}
        xml_path = cands[0]
    root = ET.parse(str(xml_path)).getroot()
    info, umpp = {"xml": Path(xml_path).name}, {}
    for sv in root.findall(".//PVStateValue"):
        k = sv.get("key")
        if k == "micronsPerPixel":
            for iv in sv.iter():
                if iv.get("index") in ("XAxis", "YAxis") and iv.get("value") is not None:
                    umpp[iv.get("index")] = float(iv.get("value"))   # Z deliberately ignored
        elif k in ("opticalZoom", "objectiveLens", "objectiveLensMag",
                   "pixelsPerLine", "linesPerFrame", "dwellTime"):
            info[k] = sv.get("value")
        elif k == "positionCurrent":
            for el in sv.iter():                       # Z depth lives one level deeper
                if el.get("index") == "ZAxis":
                    v = el.get("value")
                    if v is None:
                        sub = el.find(".//*[@value]")
                        v = sub.get("value") if sub is not None else None
                    if v is not None:
                        info["z_um"] = v
                    break
    if not umpp:
        return None, dict(info, error="micronsPerPixel absent from this XML")
    info["microns_per_pixel"] = umpp
    if len(umpp) == 2 and abs(umpp["XAxis"] - umpp["YAxis"]) > 0.02 * max(umpp.values()):
        print(f"  NOTE: non-square pixels {umpp} — using the mean; ROI areas are "
              f"approximate and axis_ratio is biased along the finer axis.")
    return float(np.mean(list(umpp.values()))), info

_um_xml, _xml_info = pv_fov_scale(DATA_PATH, XML_PATH)
um_per_px = UM_PER_PX if UM_PER_PX is not None else _um_xml
if um_per_px is None:
    raise RuntimeError(
        f"Could not determine µm/pixel ({_xml_info.get('error')}).\n"
        f"FIX (either one, in the parameters cell):\n"
        f"  UM_PER_PX = <value>   # PrairieView shows it as 'Microns per pixel'\n"
        f"  XML_PATH  = Path('/full/path/TSeries-....xml')")
print(f"XML: {_xml_info.get('xml', '(bypassed)')}   "
      f"zoom {_xml_info.get('opticalZoom', '?')}  z {_xml_info.get('z_um', '?')} um")
print(f"scale: {um_per_px:.4f} um/px"
      f"{'  (MANUAL OVERRIDE)' if UM_PER_PX is not None else ''}"
      f"   ->  FOV {X*um_per_px:.0f} x {Y*um_per_px:.0f} um")

# ---- soma diameter (um) -> accepted ROI area (px) ----
_d_px  = np.array(SOMA_DIAM_UM) / um_per_px           # diameter in pixels
SIZE_MIN, SIZE_MAX = (int(round(np.pi / 4 * _d_px[0] ** 2)),
                      int(round(np.pi / 4 * _d_px[1] ** 2)))
SIZE_GROW_CAP = int(round(SIZE_GROW_MULT * SIZE_MAX))
px2um2 = um_per_px ** 2
def area_to_diam_um(a):                               # equivalent-circle diameter
    return 2.0 * np.sqrt(np.asarray(a) * px2um2 / np.pi)

print(f"soma band: {SOMA_DIAM_UM[0]}-{SOMA_DIAM_UM[1]} um diameter"
      f"  =  {_d_px[0]:.1f}-{_d_px[1]:.1f} px across"
      f"  =  AREA {SIZE_MIN}-{SIZE_MAX} px   (grow cap {SIZE_GROW_CAP})")
print(f"v4 used 15-100 px  =  {area_to_diam_um(15):.1f}-{area_to_diam_um(100):.1f} um "
      f"equivalent diameter")
if 100 < SIZE_MIN or 15 > SIZE_MAX:
    print("  *** v4's pixel band DID NOT OVERLAP this FOV's soma band. That mismatch\n"
          "      alone explains a dendrite-only ROI set: every real soma fell outside\n"
          "      the accepted area while process fragments fell inside. ***")
elif 100 < SIZE_MAX:
    print(f"  NOTE: v4's ceiling (100 px = {area_to_diam_um(100):.1f} um) cut off the "
          f"upper {100.0*(1-(100-SIZE_MIN)/max(SIZE_MAX-SIZE_MIN,1)):.0f}% of the soma "
          f"band — the largest somata were being rejected as oversized.")
if _d_px[0] < 4:
    print(f"  WARNING: a {SOMA_DIAM_UM[0]} um soma is only {_d_px[0]:.1f} px across at "
          f"this zoom. Shape metrics need >=4-5 px; the shape gate will be unreliable.")

## Step 1 — 1-D temporal Gaussian filter (instructor code, VERBATIM)
Each pixel's time series is convolved with a 1-D Gaussian (σ = `GAUSS_WIDTH` frames).
Per-frame shot noise averages away; every frame becomes a local time-average with real
structure for the registration step. Fidelity notes: kernel and convolution are the
instructor notebook's exact code (only `gaussWidth` now reads from the parameters
cell); `ndimage.convolve` keeps uint16 (rounding ≤0.5 count — negligible at these
intensities); the 20-tap kernel's one-sample asymmetry is retained (see header).
The raw movie is deleted afterwards — everything downstream uses the filtered movie.

In [ ]:
gaussWidth = GAUSS_WIDTH
filterSize = 2*gaussWidth
pix4filt   = np.arange(-filterSize,filterSize)**2

# Temporal filters operate in 1D, so we need a 1D Gaussian bump:
gFilt1D = np.exp(-(pix4filt)/(2*(gaussWidth)**2))
gFilt1D = gFilt1D/gFilt1D.sum()
gFilt1D = np.expand_dims(np.expand_dims(gFilt1D,axis=1),axis=2)
clearFrames1D = ndimage.convolve(frames,gFilt1D)
clearFrames1D = np.array(clearFrames1D)

del frames          # raw no longer needed; frees ~0.5x movie of RAM
print(f"filtered movie: {clearFrames1D.shape} {clearFrames1D.dtype} "
      f"(sigma {gaussWidth} frames = {gaussWidth*FRAME_PERIOD*1e3:.0f} ms)")

## Step 3 (runs after MC below) — PCA denoising of the aligned movie
Same computation as the instructor notebook's SVD cell, with the v3 engineering fixes:
float32 throughout and **in-place block reconstruction** (v2 cast the movie to float64,
~57 GB at 512²×27k; here peak extra ≈ the rank-k factor, ~50 MB). Running it AFTER MC
avoids the motion-in-top-components problem that broke pre-MC PCA.

Stated once, per the team's accepted trade-off: reconstruction puts every pixel on a
shared rank-`PCA_RANK` temporal basis, which inflates pairwise ROI-trace correlations
by construction — keep this in mind when interpreting correlation/CCA magnitudes, and
consider reporting key numbers with `PCA_ENABLE = False` as a control.

In [ ]:
def pca_denoise_inplace(mov_f32, rank, chunk=2000):
    """Rank-`rank` reconstruction of (T,Y,X) float32 movie, written back IN PLACE.
    Memory-safe: the (T, Y*X) matrix is a reshape VIEW of the movie (no copy);
    only the factors + one row-block are allocated."""
    Tn, Yn, Xn = mov_f32.shape
    M = mov_f32.reshape(Tn, Yn * Xn)          # view, not a copy
    t0 = time.time()
    try:
        from sklearn.utils.extmath import randomized_svd
        U, S, Vt = randomized_svd(M, n_components=rank, random_state=0)
    except ImportError:
        from scipy.sparse.linalg import svds
        U, S, Vt = svds(M, k=rank)
    for i in range(0, Tn, chunk):
        M[i:i+chunk] = (U[i:i+chunk] * S) @ Vt
    print(f"PCA denoise rank {rank}: {time.time()-t0:.0f}s")



## Step 2 — motion correction on the FILTERED movie (two-pass rigid, v2 algorithm)
Identical two-pass structure and shift interpolation to v2, now consuming
`clearFrames1D` per the restructured ordering. **One flagged parameter deviation**
(see the comment in the next cell): `normalization=None` in the phase-correlation
call — measured 14× lower registration error on temporally-filtered frames than the
skimage default; with the default, this ordering silently loses ~1 px of accuracy.
Engineering retained from v3 (no math changes): float32 everywhere; the pass-1
aligned stack is never materialized — only its **mean** is accumulated to build the
refined template. Peak memory ≈ filtered uint16 + ONE aligned float32 stack ≈
1.5× movie-float32 (~43 GB at 512²×27k).

In [ ]:
# FLAGGED DEVIATION from v2 defaults, required by the v4 ordering (measured, not
# assumed): temporal filtering removes temporal noise but leaves per-frame SPATIAL
# high-frequency noise, and skimage's default normalization="phase" whitens the
# spectrum — upweighting exactly that noise. On ground-truth synthetic data the
# default gave 1.23 px RMS registration error on filtered frames; classic
# cross-correlation (normalization=None) gave 0.086 px (r=0.999 vs truth).
MC_NORMALIZATION = None      # <-- REQUIRED for the v4 ordering. Do not change to "phase".

def estimate_shifts(mov, template, upsample=10):
    ys, xs = np.empty(len(mov)), np.empty(len(mov))
    kw = {"upsample_factor": upsample}
    try:
        phase_cross_correlation(template, mov[0], normalization=MC_NORMALIZATION, **kw)
        kw["normalization"] = MC_NORMALIZATION
    except TypeError:              # very old skimage: kwarg absent; default applies
        print("WARNING: this skimage has no `normalization` kwarg — registration will "
              "use the phase-normalized default, which measured ~14x worse on filtered "
              "frames. Upgrade skimage if shifts look noisy.")
    for i in tqdm(range(len(mov)), desc="shifts", leave=False):
        (dy, dx), _, _ = phase_cross_correlation(template, mov[i], **kw)
        ys[i], xs[i] = dy, dx
    return ys, xs

# pass 1: estimate against the filtered-movie mean, accumulate the aligned MEAN only
filtered_anatomy = clearFrames1D.mean(0)
y1, x1 = estimate_shifts(clearFrames1D, filtered_anatomy)
acc = np.zeros((Y, X), np.float64)
for i in tqdm(range(T), desc="template", leave=False):
    acc += ndshift(clearFrames1D[i].astype(np.float32), (y1[i], x1[i]))
aligned_anatomy = (acc / T).astype(np.float32)

# pass 2: re-estimate the UNSHIFTED filtered frames against the refined template
# (no double interpolation — same structure as v2)
y2, x2 = estimate_shifts(clearFrames1D, aligned_anatomy)
total_shift_1 = np.hypot(y1, x1)
total_shift_2 = np.hypot(y2, x2)

In [ ]:
# apply pass-2 shifts -> the ONE aligned float32 stack used everywhere downstream
final_frames = np.empty((T, Y, X), np.float32)
for i in tqdm(range(T), desc="apply", leave=False):
    final_frames[i] = ndshift(clearFrames1D[i].astype(np.float32), (y2[i], x2[i]))
final_anatomy = final_frames.mean(0)
max_shift = float(np.abs(np.concatenate([y2, x2])).max())
del clearFrames1D      # aligned stack replaces it; frees ~0.5x movie
print(f"max |shift| = {max_shift:.2f} px")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(time_vector, total_shift_1, lw=.6, label="pass 1")
axes[0].plot(time_vector, total_shift_2, lw=.6, label="pass 2")
axes[0].set(xlabel="time (s)", ylabel="total shift (px)"); axes[0].legend()
for ax, img, name in ((axes[1], original_anatomy, "before"), (axes[2], final_anatomy, "after")):
    ax.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
    ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
if PCA_ENABLE:
    pca_denoise_inplace(final_frames, min(PCA_RANK, T - 1))
    final_anatomy = final_frames.mean(0)
else:
    print("PCA step skipped (PCA_ENABLE = False)")

## Temporal binning for detection (directive 5) — the discussion
**Why bin at all:** raw resonant frames are shot-noise dominated. Averaging n frames
leaves the shared calcium signal intact but cuts independent noise by √n, so the
pixel-vs-neighbors correlation — which collapsed to ~0.17 on this dataset unbinned —
recovers. (Reproduced synthetically: 0.16 unbinned → 0.40 at bin 10, cells recovered.)

**What sets the optimum:**
- *Lower bound:* enough SNR that cell pixels clear `CORR_THRESHOLD` with margin.
- *Upper bound:* the GCaMP7s transient itself (rise ~50 ms, decay τ ≈ 1–1.5 s). Bins
  much longer than ~1 s start averaging *across* transients, diluting the very signal
  correlations we detect with. Bins up to ~0.5 s are essentially free; ~1 s is fine.
- At 30.5 Hz that puts the useful range at ~5–30 frames. "auto" picks ~6 Hz (bin 5) —
  deliberately conservative; noisy data often does better at 10–20.
- Binning is DETECTION-ONLY: traces are always extracted at full rate afterwards, so
  the choice affects *which* pixels form ROIs, not the time resolution of the science.
- NOTE (v4 ordering): the movie is already low-passed by the 1-D filter (and PCA if
  enabled), so the sweep may clear threshold at small bins — pick the smallest that does.

**Empirical selection:** the sweep below computes the correlation map on a center crop
for several bin factors. Pick the smallest bin whose max clears ~0.4–0.5 — past the
knee, more binning buys little and eventually hurts.

In [ ]:
def bin_movie(mov, n):
    if n <= 1:
        return mov
    Tb = (len(mov) // n) * n
    return mov[:Tb].reshape(-1, n, *mov.shape[1:]).mean(axis=1)

def neighbor_corr_map(mov):
    """Pearson r of each pixel vs the SUM of its 8 neighbors — exact, one streaming
    pass, no movie copies (proven equal to the per-pixel pearsonr loop to 1e-6)."""
    Tn = len(mov)
    k = np.ones((3, 3))
    sx = np.zeros(mov.shape[1:]); sxx = np.zeros(mov.shape[1:])
    ss = np.zeros(mov.shape[1:]); sss = np.zeros(mov.shape[1:]); sxs = np.zeros(mov.shape[1:])
    for t in range(Tn):
        f = mov[t].astype(np.float64)
        s = convolve(f, k, mode="constant") - f
        sx += f; sxx += f * f; ss += s; sss += s * s; sxs += f * s
    num = Tn * sxs - sx * ss
    den = np.sqrt((Tn * sxx - sx**2) * (Tn * sss - ss**2))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, 0.0)   # constant pixels -> 0, never NaN

# ---- bin sweep on a center crop (fast, ~seconds each) ----
cy, cx_ = Y // 2, X // 2
crop = final_frames[:, max(0, cy-64):cy+64, max(0, cx_-64):cx_+64]
print("bin  eff.rate   corr-map max (center crop)")
for nb in [1, 2, 3, 5, 8, 12, 20, 30]:
    if nb < len(crop):
        cm = neighbor_corr_map(bin_movie(crop, nb))
        print(f"{nb:3d}  {frame_rate/nb:6.1f} Hz   {cm[2:-2, 2:-2].max():.3f}")

In [ ]:
DETECT_BIN_N = max(1, int(round(frame_rate / 6.0))) if DETECT_BIN == "auto" else int(DETECT_BIN)
detect_mov = bin_movie(final_frames, DETECT_BIN_N)
print(f"detection movie: bin x{DETECT_BIN_N} -> {len(detect_mov)} frames @ "
      f"{frame_rate/DETECT_BIN_N:.1f} Hz")
correlation_map = neighbor_corr_map(detect_mov)

## Border guard + NaN hygiene (directives 5 & 6)
Shifting fills edges with zeros in *some frames only* — edge-pixel traces become gated
by the shift time course, which is **shared across all edge pixels**, so they correlate
near 1.0 with each other and grow fake "cells" of pure motion artifact. The dead band
is exactly the maximum shift, so the border is `max(BORDER_PX_MIN, ceil(max|shift|)+1)`.
NaNs (none can arise from the streaming map, but belt-and-suspenders for any edit) would
otherwise win `argmax` and hijack ROI seeding.

In [ ]:
border = max(BORDER_PX_MIN, int(np.ceil(max_shift)) + 1)
corr_bordered = np.nan_to_num(correlation_map, nan=0.0, posinf=0.0, neginf=0.0).copy()
corr_bordered[:border, :] = 0; corr_bordered[-border:, :] = 0
corr_bordered[:, :border] = 0; corr_bordered[:, -border:] = 0
print(f"border zeroed: {border} px (max shift {max_shift:.2f})")

# ---- THRESHOLD RECALIBRATION (required by the v4 ordering) ----
# The 1-D temporal filter raises correlations EVERYWHERE, background included, so a
# threshold tuned on unfiltered data over-detects. On ground-truth synthetic data with
# this ordering: background median 0.28 / p95 0.75, true cells 0.60-0.85; threshold 0.3
# gave 16 ROIs and missed a cell, 0.5-0.6 gave 7-8 ROIs and found all three.
# RULE OF THUMB: set CORR_THRESHOLD near the map's 95th percentile, then eyeball the
# ROI overlay. The printout below gives you that number for THIS run.
_vals = corr_bordered[corr_bordered > 0]
print(f"corr map: median {np.median(_vals):.3f}  p95 {np.percentile(_vals, 95):.3f}  "
      f"max {corr_bordered.max():.3f}   <- CORR_THRESHOLD is {CORR_THRESHOLD}")
if CORR_THRESHOLD < np.percentile(_vals, 90):
    print(f"  WARNING: threshold is below the map's 90th percentile "
          f"({np.percentile(_vals, 90):.3f}) — expect over-detection of background.")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(final_anatomy, cmap="gray",
               vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
axes[0].set_title("anatomy"); axes[0].axis("off")
im = axes[1].imshow(corr_bordered, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"correlation map (max {corr_bordered.max():.2f})"); axes[1].axis("off")
plt.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()
if corr_bordered.max() < CORR_THRESHOLD:
    print(f"WARNING: corr map max {corr_bordered.max():.2f} < threshold {CORR_THRESHOLD}"
          f" — no ROIs will grow. Increase DETECT_BIN (see sweep) before proceeding.")

## ⚠️ REQUIRED BEFORE TRUSTING ROIs — recalibrate `CORR_THRESHOLD` for this dataset

**`CORR_THRESHOLD = 0.3` is inherited from the ORIGINAL (unfiltered) pipeline and is
wrong for the v4 ordering.** The 1-D temporal filter raises pixel-neighbor correlations
*everywhere* — background included — so a threshold tuned on unfiltered data
over-detects badly.

Measured on ground-truth synthetic data with this exact ordering (three planted cells):

| `CORR_THRESHOLD` | ROIs found | true cells recovered | precision |
|---|---|---|---|
| 0.30 (inherited) | 16 | 2 / 3 | 0.12 |
| **0.50** | 8 | **3 / 3** | 0.38 |
| **0.60** | 7 | **3 / 3** | 0.43 |
| 0.70 | 5 | 2 / 3 | 0.40 |
| 0.80 | 0 | 0 / 3 | — |

In that run the correlation map's background sat at median 0.28 / p95 0.75 while true
cells were 0.60–0.85 — i.e. the useful threshold tracked the map's **~95th percentile**.

**Procedure for each dataset (do this once per FOV/imaging condition, not per run):**
1. Read the `corr map: median / p95 / max` printout from the cell above.
2. Set `CORR_THRESHOLD` near that **p95** value, re-run from the parameters cell.
3. Look at the ROI overlay below. Too many blobs in obviously empty tissue → raise it;
   visible cells missing → lower it. Record the final value in the run notes.
4. The cell above prints a WARNING when the threshold is below the map's 90th
   percentile — treat that warning as "the ROI count is not trustworthy yet".

The defaults here (`CORR_THRESHOLD = 0.5`, `DETECT_BIN = 1`) are the calibrated
starting point for filtered data, **not** validated values for your FOV.

## Data-driven check on the size band (v5)
The band above comes from the XML scale plus a biological prior. This cell asks the
**image itself** how big its bright objects are, as an independent check that the
prior is not absurd for this FOV.

Method: high-pass the aligned anatomy (removes slow shading), take its 2-D
autocorrelation, and read the radius at which the radial profile falls to half. For a
field of filled disks of diameter *d*, that half-radius sits at ≈ 0.404·*d*, so
*d* ≈ 2.48 × half-radius. It is a **dominant-scale** estimate, good to roughly ±30%
and biased *small* when thin processes dominate the image — which is itself the
diagnostic: a dominant scale far below the soma band means the bright structure in
this FOV really is mostly neuropil and processes, and no acceptance rule will
manufacture somata that were not imaged.

Read it as a sanity check, not as a parameter. If it lands wildly outside the declared
band, look at the anatomy image before changing anything.

*Measured on the ground-truth synthetic (somata of ~8.8 px FWHM plus five processes):
it reported 7.4 px — 16% low, in the direction the processes predict.*

In [ ]:
def dominant_feature_diam_px(img, max_lag=60):
    a = img.astype(np.float64)
    a = a - ndimage.gaussian_filter(a, max(3.0, max_lag / 4.0))   # kill slow shading
    a = a - a.mean()
    F = np.fft.rfft2(a)
    ac = np.fft.fftshift(np.fft.irfft2(F * np.conj(F), s=a.shape))
    ac = ac / ac.max()
    cy, cx = np.array(ac.shape) // 2
    yy, xx = np.ogrid[:ac.shape[0], :ac.shape[1]]
    rr = np.hypot(yy - cy, xx - cx).astype(int)
    n = int(min(max_lag, rr.max()))
    prof = ndimage.mean(ac, labels=rr, index=np.arange(n))
    # Normalise by lag 1, NOT lag 0. Shot noise is uncorrelated between pixels, so it
    # contributes a delta spike at lag 0 ONLY; normalising by lag 0 measures the noise
    # scale (~1 px) instead of the object scale. Measured on ground truth: lag-0
    # normalisation reported 5 px for 12 px cells; lag-1 normalisation recovers them.
    if n < 3 or not np.isfinite(prof[1]) or prof[1] <= 0:
        return float("nan"), prof
    prof = prof / prof[1]
    below = np.where(prof[1:] < 0.5)[0]
    half = float(below[0] + 1) if len(below) else float("nan")
    return 2.48 * half, prof          # 0.404*d is the half-radius for a filled disk

_dom_px, _prof = dominant_feature_diam_px(final_anatomy)
print(f"dominant bright-feature diameter (autocorrelation): {_dom_px:.1f} px "
      f"= {_dom_px*um_per_px:.1f} um   vs declared soma band "
      f"{SOMA_DIAM_UM[0]}-{SOMA_DIAM_UM[1]} um")
if np.isfinite(_dom_px) and _dom_px * um_per_px < 0.5 * SOMA_DIAM_UM[0]:
    print("  NOTE: dominant scale is well below the soma band — this FOV is dominated "
          "by fine structure (processes/neuropil). Expect few accepted somata; check "
          "the anatomy image and the imaging depth before loosening the gate.")

plt.figure(figsize=(4, 3))
plt.plot(np.arange(len(_prof)) * um_per_px, _prof, lw=1)
plt.axhline(.5, color="r", ls="--", lw=.8)
plt.axvspan(SOMA_DIAM_UM[0] * .404, SOMA_DIAM_UM[1] * .404, color="g", alpha=.15,
            label="soma band (half-radius)")
plt.xlabel("lag (um)"); plt.ylabel("autocorr"); plt.legend(fontsize=7)
plt.tight_layout(); plt.show()

## ROI extraction (v5) — data-driven, indexing-safe, **soma-selective**
### v5 addition: the shape gate
Growth and detection are unchanged; **acceptance** now tests shape as well as area.
Two metrics, computed from each candidate mask, deliberately complementary:

| metric | what it is | round soma | dendritic process |
|---|---|---|---|
| `axis_ratio` | major/minor axis of the equivalent ellipse (2nd moments of the pixel coordinates, with the standard +1/12 pixel-quantisation correction) | 1.0–1.6 | **>3** for a straight segment |
| `solidity` | mask area ÷ convex-hull area | 0.85–0.95 | **<0.6** when curved or branched |

Neither alone is sufficient: a perfectly **straight** thin process has solidity ≈ 1
(its hull is just the enclosing rectangle) and is caught only by `axis_ratio`; a
**bent or branching** process can have a modest axis ratio and is caught only by
`solidity`. Requiring both is what makes the gate specific.

Rejection is safe for the loop: `next_roi` has already zeroed the candidate's pixels
in `used_map` before acceptance is tested, so a rejected ROI consumes its seed and the
scan continues — termination is unaffected.

**Validated on ground truth** (synthetic FOV, 5 somata of 10–13 µm + 5 dendritic
processes, matched shot noise and drift): with the gate off, acceptance returned
**5 somata and 5 processes**; with the gate on, **5 somata and 0 processes** — no
soma lost. Under `PCA_ENABLE = True` the result was identical.

### What this gate does NOT fix — read before trusting the ROI set
The gate rejects objects that are *shaped* like processes. It cannot reject a
compact, round blob of **neuropil** that happens to correlate well, because such a
blob is shaped exactly like a soma. Neuropil contamination is controlled by
`CORR_THRESHOLD` and by looking at the overlay — not by shape. Concretely:
- **Elongated / branched ROIs** → this gate handles it.
- **Round ROIs sitting on featureless tissue** → raise `CORR_THRESHOLD`, and set
  `PCA_ENABLE = False` (the shared rank-`PCA_RANK` basis inflates long-range
  correlations, which lets growth run further and spread through neuropil).
- **No somata anywhere in the FOV** → an acquisition fact, not a parameter. Check the
  imaging depth printed by the FOV-scale cell: superficial planes are apical dendrite
  fields and contain few or no cell bodies.

Extraction slow? Lower `SIZE_GROW_MULT` — it sets how far an oversized object is
allowed to grow before being judged, and growth cost scales with it.

### Retained from v3/v4, each load-bearing:
1. **No `n_rois` hardcode.** Extraction runs until the correlation map is *exhausted*
   (no remaining seed above `CORR_THRESHOLD`) — the data decides the count. A cap of
   `MAX_CANDIDATES` exists purely as a runaway guard (each attempt consumes ≥1 seed
   pixel, so termination is guaranteed regardless).
2. **`this_trace = ....copy()`** — in v2 this was a VIEW into the movie, and `+=`
   silently wrote the growing ROI sum back into `final_frames`, corrupting every later
   ROI's correlations. This was the most dangerous bug in the notebook.
3. **Labels == rows.** v2 stamped `roi_map` with the attempt index but compacted the
   trace array — map label k did not index trace row k. Here masks are collected and
   labeled 1..n in exactly trace-row order.
4. Growth runs on the detection movie (`DETECT_BIN`; with the v4 ordering that is
   normally the filtered movie itself, bin=1); traces come from the **full-rate**
   movie in the next cell.
5. `pearsonr` replaced by the identical dot-product formula (scale-invariant Pearson,
   equal to scipy to 1e-10) — turns hours into minutes at full FOV.

**Check the ROI overlay against `CORR_THRESHOLD` before trusting the count** — with the
filtered ordering the threshold is the single most sensitive parameter (see the
recalibration printout above).

In [ ]:
def fast_pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else 0.0

def next_roi(corr_map, mov, corr_threshold, grow_cap):
    i, j = np.unravel_index(np.argmax(corr_map), corr_map.shape)
    this_trace = mov[:, i, j].astype(np.float64).copy()      # COPY — fix #2 above
    this_roi = np.zeros(corr_map.shape, np.uint8)
    this_roi[i, j] = 1
    used = corr_map.copy(); used[i, j] = 0
    growing = True
    while growing and this_roi.sum() < grow_cap:
        growing = False
        ring = np.argwhere(binary_dilation(this_roi, np.ones((3, 3))) ^ this_roi.astype(bool))
        add = np.zeros(len(this_trace))
        for (yy, xx) in ring:
            if used[yy, xx] != 0:
                px = mov[:, yy, xx].astype(np.float64)
                if fast_pearson(this_trace, px) > corr_threshold:
                    growing = True
                    this_roi[yy, xx] = 1
                    used[yy, xx] = 0
                    add += px
        this_trace += add
    return this_roi, this_roi.sum(), used

def roi_shape(mask):
    """(area_px, axis_ratio, solidity) for a boolean mask — v5.

    axis_ratio: sqrt of the eigenvalue ratio of the pixel-coordinate covariance,
      i.e. major/minor axis of the equivalent ellipse. The +1/12 term is the standard
      correction for pixel quantisation: without it a 1-px-wide line has zero minor
      variance and the ratio blows up to infinity instead of a finite, comparable
      number. Filled disk -> 1.0 exactly; 1x30 px line -> 30.
    solidity: area / convex-hull area. Hull is taken over pixel CORNERS (+-0.5) so a
      small filled blob scores ~1.0 rather than being penalised for discretisation.
    """
    ys, xs = np.nonzero(mask)
    a = int(len(ys))
    p = np.stack([ys, xs], 1).astype(np.float64)
    c = p - p.mean(0)
    cov = (c.T @ c) / max(a - 1, 1) + np.eye(2) / 12.0
    ev = np.sort(np.linalg.eigvalsh(cov))
    axis_ratio = float(np.sqrt(max(ev[1], 1e-12) / max(ev[0], 1e-12)))
    solidity = 1.0                      # neutral: never rejects when unmeasurable
    if a >= 3 and _HULL is not None:
        try:
            corners = np.concatenate([p + o for o in
                                      ((-.5, -.5), (-.5, .5), (.5, -.5), (.5, .5))])
            solidity = float(a / max(_HULL(corners).volume, 1e-9))
        except Exception:               # degenerate/collinear hull
            pass
    return a, axis_ratio, solidity

masks, cand = [], []
used_map = corr_bordered.copy()
t0 = time.time()
while used_map.max() > CORR_THRESHOLD and len(cand) < MAX_CANDIDATES:
    roi, size, used_map = next_roi(used_map, detect_mov, CORR_THRESHOLD, SIZE_GROW_CAP)
    m = roi.astype(bool)
    a, ar, sol = roi_shape(m)
    # NOTE (flagged deviation): v4 used strict `SIZE_MIN < size < SIZE_MAX`; v5 is
    # inclusive, because the bounds are now derived quantities rather than round
    # numbers and excluding the endpoint would be arbitrary. Effect: <=2 ROIs.
    ok_size  = SIZE_MIN <= a <= SIZE_MAX
    ok_shape = (not SHAPE_FILTER) or (ar <= MAX_AXIS_RATIO and sol >= MIN_SOLIDITY)
    cand.append({"area": a, "axis_ratio": ar, "solidity": sol,
                 "ok_size": bool(ok_size), "ok_shape": bool(ok_shape), "mask": m})
    if ok_size and ok_shape:
        masks.append(m)

sizes_all = [c["area"] for c in cand]
n_small  = sum(1 for c in cand if c["area"] < SIZE_MIN)
n_big    = sum(1 for c in cand if c["area"] > SIZE_MAX)
n_elong  = sum(1 for c in cand if c["ok_size"] and c["axis_ratio"] > MAX_AXIS_RATIO)
n_ragged = sum(1 for c in cand if c["ok_size"] and c["axis_ratio"] <= MAX_AXIS_RATIO
                                  and c["solidity"] < MIN_SOLIDITY)
print(f"{len(cand)} candidates grown in {time.time()-t0:.0f}s -> {len(masks)} accepted")
print(f"  rejected: {n_small} too small (<{SIZE_MIN} px)   {n_big} too large "
      f"(>{SIZE_MAX} px)   {n_elong} elongated (axis_ratio>{MAX_AXIS_RATIO})   "
      f"{n_ragged} ragged (solidity<{MIN_SOLIDITY})")
print(f"  candidate areas px min/med/max "
      f"{min(sizes_all)}/{int(np.median(sizes_all))}/{max(sizes_all)}")
if len(cand) >= MAX_CANDIDATES:
    print(f"  *** MAX_CANDIDATES ({MAX_CANDIDATES}) HIT — the correlation map was NOT\n"
          f"      exhausted and this ROI set is TRUNCATED. Raise CORR_THRESHOLD (the\n"
          f"      usual cause) or raise the cap. Do not report this run as-is. ***")
_tiny = sum(1 for a in sizes_all if a <= 2)
if _tiny > 0.5 * len(sizes_all):
    print(f"  NOTE: {_tiny}/{len(sizes_all)} candidates were <=2 px — seeds that grew "
          f"nothing.\n        That is the signature of CORR_THRESHOLD set too LOW; it "
          f"costs time, not correctness.")
if masks:
    _d = area_to_diam_um([int(m.sum()) for m in masks])
    print(f"  ACCEPTED ROI diameter: median {np.median(_d):.1f} um "
          f"(IQR {np.percentile(_d,25):.1f}-{np.percentile(_d,75):.1f}), "
          f"n = {len(masks)}")
else:
    print("  *** 0 ROIs accepted. Check the diagnostic plots below BEFORE loosening "
          "the gate — 0 accepted with many elongated rejects means the FOV really is "
          "process-dominated. ***")

roi_map = np.zeros((Y, X), np.uint16)
for k, m in enumerate(masks):
    roi_map[m] = k + 1        # label k+1 == trace row k, ALWAYS  (fix #3)

# per-accepted-ROI shape record, in trace-row order (saved with the traces)
_ok = [c for c in cand if c["ok_size"] and c["ok_shape"]]
roi_axis_ratio = np.array([c["axis_ratio"] for c in _ok], np.float32)
roi_solidity   = np.array([c["solidity"]   for c in _ok], np.float32)
assert len(_ok) == len(masks), 'accepted-metric list desynced from masks'

## Shape-gate diagnostics (v5) — see what was cut, and why
Three views of the same decision, so the cutoffs are chosen from *this* data rather
than inherited:
1. **area vs elongation** and **elongation vs solidity** — every grown candidate, the
   accept box drawn on top. Somata should form a cloud near the bottom-left of the
   second panel; processes stream out along the elongation axis.
2. **the anatomy overlay** — accepted in green, shape-rejected in red. If the red
   objects trace visible processes and the green ones sit on visible cell bodies, the
   gate is doing its job. If green blobs sit on neuropil, raise `CORR_THRESHOLD`.

Move `MAX_AXIS_RATIO` / `MIN_SOLIDITY` to the visible gap between the clouds. Record
the values you settle on — they are a reportable methods number.

In [ ]:
if cand:
    A   = np.array([c["area"] for c in cand], float)
    AR  = np.array([c["axis_ratio"] for c in cand])
    SOL = np.array([c["solidity"] for c in cand])
    acc = np.array([c["ok_size"] and c["ok_shape"] for c in cand])

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    ax[0].scatter(area_to_diam_um(A[~acc]), AR[~acc], s=14, c="0.6", label="rejected")
    ax[0].scatter(area_to_diam_um(A[acc]),  AR[acc],  s=18, c="tab:green", label="accepted")
    ax[0].axvspan(SOMA_DIAM_UM[0], SOMA_DIAM_UM[1], color="tab:green", alpha=.10)
    ax[0].axhline(MAX_AXIS_RATIO, color="r", ls="--", lw=.8)
    ax[0].set(xlabel="equivalent diameter (um)", ylabel="axis ratio", yscale="log")
    ax[0].legend(fontsize=7)

    ax[1].scatter(AR[~acc], SOL[~acc], s=14, c="0.6")
    ax[1].scatter(AR[acc],  SOL[acc],  s=18, c="tab:green")
    ax[1].axvline(MAX_AXIS_RATIO, color="r", ls="--", lw=.8)
    ax[1].axhline(MIN_SOLIDITY,  color="r", ls="--", lw=.8)
    ax[1].set(xlabel="axis ratio (log)", ylabel="solidity", xscale="log")
    ax[1].set_title("accept = left of AND above both lines")

    ax[2].imshow(final_anatomy, cmap="gray",
                 vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
    rej_map = np.zeros((Y, X), bool)
    for c in cand:
        if not (c["ok_size"] and c["ok_shape"]):
            rej_map |= c["mask"]
    ax[2].imshow(np.ma.masked_where(~rej_map, rej_map), cmap="autumn", alpha=.45)
    ax[2].imshow(np.ma.masked_where(roi_map == 0, roi_map > 0), cmap="winter", alpha=.55)
    ax[2].set_title(f"green = {len(masks)} accepted, red = "
                    f"{len(cand)-len(masks)} rejected"); ax[2].axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
# traces at FULL rate from the aligned movie (sum over member pixels; identical
# semantics to the grown sum, order-independent)
if masks:
    traces_raw = np.stack([final_frames[:, m].sum(axis=1) for m in masks]).astype(np.float32)
    roi_npix = np.array([int(m.sum()) for m in masks])
else:
    traces_raw = np.zeros((0, T), np.float32); roi_npix = np.array([], int)
print("traces_raw:", traces_raw.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(final_anatomy, cmap="gray",
               vmin=np.percentile(final_anatomy, 1), vmax=np.percentile(final_anatomy, 99))
mm = np.ma.masked_where(roi_map == 0, roi_map)
axes[0].imshow(mm, cmap="prism", alpha=.45); axes[0].set_title(f"{len(masks)} ROIs"); axes[0].axis("off")
if len(traces_raw):
    tz = np.nan_to_num(zscore(traces_raw, axis=1))   # constant traces -> 0, not NaN
    axes[1].imshow(tz[np.argsort(np.argmax(tz, 1))], aspect="auto", cmap="afmhot",
                   vmin=0, vmax=np.percentile(tz, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(tz) + 1])
    axes[1].set(xlabel="time (s)", ylabel="ROI # (viz only: z-scored)")
plt.tight_layout(); plt.show()

## ΔF/F (static bottom-15% F0, team decision) + bleaching check (directive 8)
Static F0 preserves slow arousal-state differences (the science) but **assumes no strong
photobleaching** — under bleach, F0 sits near the late-run floor and inflates early ΔF/F.
So we measure it: a transient-resistant linear fit — the FOV-mean raw F is first reduced
to 1-second MEDIANS (calcium transients barely move a median), then fit with least
squares. If the fitted drift across the run exceeds `BLEACH_WARN_PCT` of the mean, this
run gets flagged and the F0 strategy revisited (per-ROI detrended F0 is the usual
remedy — decide as a team, not silently).

In [ ]:
if len(traces_raw):
    F0 = np.percentile(traces_raw, DFF_PERCENTILE, axis=1, keepdims=True)  # 15th, per ROI
    if (F0 <= 0).any():
        bad = np.where(F0.squeeze() <= 0)[0]
        print(f"WARNING: non-positive F0 for ROI rows {list(bad)} — their dF/F is "
              f"unreliable (denoised/edge traces?). Inspect before using.")
    dff = (traces_raw - F0) / np.maximum(F0, 1e-6)

    # ---- bleaching check (transient-resistant: fit 1-s MEDIANS, not raw meanF) ----
    meanF = traces_raw.mean(axis=0)
    per_sec = max(1, int(round(frame_rate)))
    nblk = len(meanF) // per_sec
    mF_1s = np.median(meanF[:nblk * per_sec].reshape(nblk, per_sec), axis=1)
    t_1s = time_vector[:nblk * per_sec].reshape(nblk, per_sec).mean(axis=1)
    slope, intercept = np.polyfit(t_1s, mF_1s, 1)
    drift_pct = 100.0 * slope * (time_vector[-1] - time_vector[0]) / meanF.mean()
    bleach_flag = abs(drift_pct) > BLEACH_WARN_PCT
    print(f"mean-F drift over run: {drift_pct:+.1f}%  "
          f"{'*** BLEACH FLAG — revisit F0 strategy ***' if bleach_flag else '(ok)'}")

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].imshow(dff[np.argsort(np.argmax(dff, 1))], aspect="auto", cmap="afmhot",
                   vmin=0, vmax=np.percentile(dff, 99),
                   extent=[time_vector[0], time_vector[-1], 1, len(dff) + 1])
    axes[0].set(ylabel="ROI #", title="dF/F")
    axes[1].plot(time_vector, meanF, lw=.6, label="FOV-mean raw F")
    axes[1].plot(time_vector, intercept + slope * time_vector, "r--",
                 label=f"drift {drift_pct:+.1f}%")
    axes[1].set(xlabel="time (s)", ylabel="mean F"); axes[1].legend()
    plt.tight_layout(); plt.show()

## ⚠️ REQUIRED CONTROL RUN — repeat this notebook with `PCA_ENABLE = False`

PCA reconstruction projects every pixel onto a **shared rank-`PCA_RANK` temporal
basis**. Any noise landing in the retained components becomes noise *shared between
ROIs*, so **pairwise ROI-trace correlations are inflated by construction**. This
project's headline analyses — pairwise correlation structure, population PCA, CCA
against arousal, GLM coupling strength — are exactly the statistics that inflate.

**So every number that reaches a slide needs a PCA-off counterpart:**
1. Run the whole notebook once with `PCA_ENABLE = True` (detection benefits from the
   cleaner movie).
2. Run it again with `PCA_ENABLE = False`, saving to a separate output folder
   (change `OUT_ROOT` or the subfolder name so the two do not overwrite each other).
3. Compare: ROI count, and the magnitude of any correlation/CCA/GLM statistic.
   - Similar values → the result is robust; report the PCA-on version if it is cleaner.
   - Materially larger with PCA on → **report the PCA-off numbers**, and say in the
     talk that denoising inflates correlation magnitudes.

Rule of thumb for the sprint: **ROI detection may use PCA; reported statistics should
come from PCA-off traces** unless the two agree. `traces_raw` is saved in both cases,
so this comparison costs one re-run, not a re-analysis.

## Save (minimal, crash-fixed — full output routing is deferred by team decision)
Raw traces are saved alongside ΔF/F: z-scoring/normalization choices stay revisable.
`params.json` records `pca_enable` and `corr_threshold`, so the PCA-on/PCA-off pair and
the calibrated threshold are always recoverable from the output folder itself.

In [ ]:
OUT_ROOT = Path('/grid/courses/data/imagcourse/GECI Project Jonathons/Data to Analyze')
out = OUT_ROOT / "neural data output" / DATA_PATH.name    # per-t-series subfolder: no overwrites
out.mkdir(parents=True, exist_ok=True)
np.save(out / "traces_raw.npy", traces_raw)
np.save(out / "roi_npix.npy", roi_npix)
np.save(out / "roi_area_um2.npy", (roi_npix * px2um2).astype(np.float32))   # v5
np.save(out / "roi_diam_um.npy", area_to_diam_um(roi_npix).astype(np.float32))
np.save(out / "roi_axis_ratio.npy", roi_axis_ratio)
np.save(out / "roi_solidity.npy", roi_solidity)
if len(traces_raw):
    np.save(out / "dff.npy", dff)
    np.save(out / "F0.npy", F0.squeeze())
np.save(out / "shifts_yx.npy", np.stack([y2, x2]))
tiff.imwrite(out / "roi_map.tif", roi_map)
(out / "params.json").write_text(json.dumps({
    "pipeline_order": "1d_gauss_filter -> motion_correct -> pca -> detect -> extract",
    "frame_period": FRAME_PERIOD, "gauss_width_frames": GAUSS_WIDTH,
    "pca_enable": PCA_ENABLE, "pca_rank": PCA_RANK, "detect_bin": DETECT_BIN_N,
    "corr_threshold": CORR_THRESHOLD, "size_min": SIZE_MIN, "size_max": SIZE_MAX,
    "size_grow_cap": SIZE_GROW_CAP, "border_px": border,
    "dff_percentile": DFF_PERCENTILE, "n_rois": len(masks), "max_shift_px": max_shift,
    # ---- v5: anatomical / soma-selective acceptance ----
    "um_per_px": um_per_px, "um_per_px_source": "manual" if UM_PER_PX is not None else "xml",
    "xml_info": _xml_info, "soma_diam_um": list(SOMA_DIAM_UM),
    "shape_filter": SHAPE_FILTER, "max_axis_ratio": MAX_AXIS_RATIO,
    "min_solidity": MIN_SOLIDITY, "n_candidates": len(cand),
    "max_candidates": MAX_CANDIDATES, "candidates_truncated": len(cand) >= MAX_CANDIDATES,
    "rejected": {"too_small": n_small, "too_large": n_big,
                 "elongated": n_elong, "ragged": n_ragged},
}, indent=2))
print("saved ->", out)